**Course Code:** 515513

**Group number:** 515513_07

**Student Name:** 呂泰廷 吳瑞傑 郭宗睿

**Student ID:** 112652030 112652027 111652043

# Task 1: Within-Subject EEG Classification

This notebook implements the Part 1 Task 1 pipeline required by the specification.

Task 1 uses the released fixed split for one subject:
- `data/train.npz` contains labeled trials.
- `data/test.npz` contains official test trials.

The goal is to compare how frequency-band choices affect classification quality. To support the report requirement, this notebook compares two preprocessing designs under the same validation protocol and model family.


## Method Summary

We use a compact feature-based classifier instead of a high-capacity neural network because Task 1 has only 16 labeled training trials. The pipeline is:

1. Select one required frequency band.
2. Apply one preprocessing design.
3. Extract log channel-variance and one-vs-rest CSP log-variance features.
4. Train a regularized LogisticRegression classifier.
5. Evaluate with 4-fold stratified validation.
6. Train the selected final pipeline on all labeled data and export a Kaggle CSV.

The official test labels are never used for preprocessing, model selection, or validation.


## Setup

Run this notebook from `part1/task1`. All paths are relative to this directory, as required by the specification.


In [ ]:
from __future__ import annotations

from pathlib import Path
import csv
import random

import numpy as np
from scipy import linalg, signal
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
DATA_DIR = Path('data')
TRAIN_FILE = DATA_DIR / 'train.npz'
TEST_FILE = DATA_DIR / 'test.npz'

BANDS = {
    'mu_8_13': (8.0, 13.0),
    'beta_13_30': (13.0, 30.0),
    'broad_4_40': (4.0, 40.0),
    'high_gamma_70_125': (70.0, 125.0),
}

PREPROCESSING_DESIGNS = {
    'A_full_trial_global_zscore': {
        'description': 'Full 4.5s trial, bandpass filtering, then one global train-set z-score.',
        'time_window': None,
        'normalization': 'global',
    },
    'B_motor_window_channel_zscore': {
        'description': 'Crop to 1.0-4.0s motor-response window, bandpass filtering, then per-channel train-set z-score.',
        'time_window': (1.0, 4.0),
        'normalization': 'channel',
    },
}

CLASS_LABELS = (0, 1, 2, 3)
C_VALUES = (0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0)
CSP_PAIRS = 1
N_SPLITS = 4


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed()

if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError('Expected data/train.npz and data/test.npz. Run this notebook from part1/task1.')

print('OK: required Task 1 files found')


## Load Data

The training set has 16 labeled trials, balanced across the four classes. This is why the validation protocol must be careful: a single random split can be too noisy, so we use 4-fold stratified validation.


In [ ]:
def load_train_data(path: Path):
    arrays = np.load(path, allow_pickle=True)
    missing = [key for key in ('x', 'y') if key not in arrays]
    if missing:
        raise KeyError(f'Missing keys in {path}: {missing}')
    sfreq = float(arrays['sfreq']) if 'sfreq' in arrays else 250.0
    return arrays['x'].astype(np.float32), arrays['y'].astype(np.int64), sfreq


def load_test_data(path: Path):
    arrays = np.load(path, allow_pickle=True)
    if 'x' not in arrays:
        raise KeyError(f"Missing key 'x' in {path}")
    x = arrays['x'].astype(np.float32)
    ids = arrays['id'].astype(np.int64) if 'id' in arrays else np.arange(len(x), dtype=np.int64)
    sfreq = float(arrays['sfreq']) if 'sfreq' in arrays else 250.0
    return x, ids, sfreq


x_train_raw, y_train, sfreq_train = load_train_data(TRAIN_FILE)
x_test_raw, test_ids, sfreq_test = load_test_data(TEST_FILE)

if abs(sfreq_train - sfreq_test) > 1e-6:
    raise ValueError(f'Sampling-rate mismatch: train={sfreq_train}, test={sfreq_test}')

print('train:', x_train_raw.shape, y_train.shape, 'sfreq:', sfreq_train)
print('test :', x_test_raw.shape, test_ids.shape, 'sfreq:', sfreq_test)
print('class counts:', dict(zip(*np.unique(y_train, return_counts=True))))


## Preprocessing Designs

Both designs use the selected frequency band, but they differ in the time region and normalization statistics.

- **Design A** keeps the full trial and uses one global z-score fitted on the training fold.
- **Design B** crops to the central motor-response window and normalizes each channel separately using training-fold statistics.

In both designs, validation and test data are transformed using statistics fitted only on the corresponding training data.


In [ ]:
def crop_time_window(x: np.ndarray, sfreq: float, time_window: tuple[float, float] | None) -> np.ndarray:
    if time_window is None:
        return x
    start_s, end_s = time_window
    start = max(int(round(start_s * sfreq)), 0)
    end = min(int(round(end_s * sfreq)), x.shape[-1])
    if start >= end:
        raise ValueError(f'Invalid time window {time_window}')
    return x[..., start:end]


def bandpass_trials(x: np.ndarray, sfreq: float, band: tuple[float, float], order: int = 4) -> np.ndarray:
    low_hz, high_hz = band
    nyquist = sfreq * 0.5
    low = max(low_hz / nyquist, 1e-5)
    high = min(high_hz / nyquist, 0.999)
    if not 0.0 < low < high < 1.0:
        raise ValueError(f'Invalid band {band} for sfreq={sfreq}')
    sos = signal.butter(order, [low, high], btype='bandpass', output='sos')
    return signal.sosfiltfilt(sos, x, axis=-1).astype(np.float32)


def fit_preprocess_transform(
    x_fit: np.ndarray,
    x_apply: np.ndarray,
    sfreq: float,
    band: tuple[float, float],
    design: dict,
) -> tuple[np.ndarray, np.ndarray]:
    x_fit = crop_time_window(x_fit, sfreq, design['time_window'])
    x_apply = crop_time_window(x_apply, sfreq, design['time_window'])
    x_fit = bandpass_trials(x_fit, sfreq, band)
    x_apply = bandpass_trials(x_apply, sfreq, band)

    if design['normalization'] == 'global':
        mean = x_fit.mean()
        std = x_fit.std() + 1e-8
    elif design['normalization'] == 'channel':
        mean = x_fit.mean(axis=(0, 2), keepdims=True)
        std = x_fit.std(axis=(0, 2), keepdims=True) + 1e-8
    else:
        raise ValueError(f"Unsupported normalization: {design['normalization']}")

    return ((x_fit - mean) / std).astype(np.float32), ((x_apply - mean) / std).astype(np.float32)


## Feature Extraction And Classifier

The feature extractor combines two EEG features:

- log channel variance, which captures band-specific signal power per channel;
- one-vs-rest CSP log-variance, which captures spatial patterns that separate each class from the others.

The classifier is LogisticRegression with `class_weight='balanced'`. We also use balanced assignment at prediction time because the released Task 1 training set is exactly balanced and the test set has the same number of trials.


In [ ]:
def normalized_covariances(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    centered = x - x.mean(axis=-1, keepdims=True)
    cov = np.matmul(centered, np.swapaxes(centered, -1, -2))
    cov /= max(centered.shape[-1] - 1, 1)
    trace = np.trace(cov, axis1=1, axis2=2)[:, None, None]
    return cov / (trace + eps)


def shrink_covariance(cov: np.ndarray, shrinkage: float = 0.20) -> np.ndarray:
    scale = np.trace(cov) / cov.shape[0]
    return (1.0 - shrinkage) * cov + shrinkage * scale * np.eye(cov.shape[0], dtype=cov.dtype)


def fit_csp_filters(x: np.ndarray, y: np.ndarray, csp_pairs: int = CSP_PAIRS) -> np.ndarray:
    covariances = normalized_covariances(x)
    filters = []
    for label in CLASS_LABELS:
        positive = covariances[y == label]
        negative = covariances[y != label]
        if len(positive) == 0 or len(negative) == 0:
            raise ValueError(f'Cannot fit CSP for missing label {label}')
        cov_pos = shrink_covariance(positive.mean(axis=0))
        cov_neg = shrink_covariance(negative.mean(axis=0))
        values, vectors = linalg.eigh(cov_pos, cov_pos + cov_neg)
        order = np.argsort(values)
        selected = np.r_[order[:csp_pairs], order[-csp_pairs:]]
        filters.append(vectors[:, selected].T.astype(np.float32))
    return np.vstack(filters)


def extract_features(x: np.ndarray, csp_filters: np.ndarray | None = None) -> np.ndarray:
    blocks = [np.log(np.var(x, axis=-1) + 1e-8)]
    if csp_filters is not None:
        projected = np.einsum('fc,nct->nft', csp_filters, x, optimize=True)
        csp_var = np.var(projected, axis=-1) + 1e-8
        csp_var /= csp_var.sum(axis=1, keepdims=True) + 1e-8
        blocks.append(np.log(csp_var))
    return np.concatenate(blocks, axis=1).astype(np.float32)


def build_classifier(c_value: float) -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(C=c_value, class_weight='balanced', max_iter=3000, random_state=SEED)),
    ])


def balanced_assignment(scores: np.ndarray, labels: tuple[int, ...] = CLASS_LABELS) -> np.ndarray:
    if scores.shape[0] % len(labels) != 0:
        raise ValueError('Balanced assignment expects the number of samples to be divisible by the number of classes.')
    quota = scores.shape[0] // len(labels)
    slots = np.asarray([label for label in labels for _ in range(quota)], dtype=np.int64)
    cost = -scores[:, slots]
    row_index, col_index = linear_sum_assignment(cost)
    pred = np.empty(scores.shape[0], dtype=np.int64)
    pred[row_index] = slots[col_index]
    return pred


## Validation Protocol

We use 4-fold stratified validation. Since each class has four labeled trials, each fold holds out one trial from each class. This gives a fairer estimate than a single random split and keeps all classes represented in every validation fold.


In [ ]:
def evaluate_design_band(design_name: str, design: dict, band_name: str, band: tuple[float, float]) -> list[dict]:
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    rows = []

    for c_value in C_VALUES:
        y_true_all = []
        raw_pred_all = []
        balanced_pred_all = []
        fold_scores = []

        for fold_id, (train_idx, val_idx) in enumerate(splitter.split(x_train_raw, y_train), start=1):
            x_fit, x_val = fit_preprocess_transform(
                x_train_raw[train_idx],
                x_train_raw[val_idx],
                sfreq_train,
                band,
                design,
            )
            csp_filters = fit_csp_filters(x_fit, y_train[train_idx])
            train_features = extract_features(x_fit, csp_filters)
            val_features = extract_features(x_val, csp_filters)

            classifier = build_classifier(c_value)
            classifier.fit(train_features, y_train[train_idx])
            probabilities = classifier.predict_proba(val_features)
            raw_pred = probabilities.argmax(axis=1)
            balanced_pred = balanced_assignment(np.log(np.clip(probabilities, 1e-12, 1.0)))

            y_true = y_train[val_idx]
            y_true_all.extend(y_true.tolist())
            raw_pred_all.extend(raw_pred.tolist())
            balanced_pred_all.extend(balanced_pred.tolist())
            fold_scores.append(f1_score(y_true, balanced_pred, labels=list(CLASS_LABELS), average='macro'))

        rows.append({
            'design': design_name,
            'band': band_name,
            'low_hz': band[0],
            'high_hz': band[1],
            'c': c_value,
            'raw_macro_f1': f1_score(y_true_all, raw_pred_all, labels=list(CLASS_LABELS), average='macro'),
            'raw_accuracy': accuracy_score(y_true_all, raw_pred_all),
            'balanced_macro_f1': f1_score(y_true_all, balanced_pred_all, labels=list(CLASS_LABELS), average='macro'),
            'balanced_accuracy': accuracy_score(y_true_all, balanced_pred_all),
            'mean_fold_balanced_macro_f1': float(np.mean(fold_scores)),
            'std_fold_balanced_macro_f1': float(np.std(fold_scores)),
        })
    return rows


def rank_results(rows: list[dict]) -> list[dict]:
    return sorted(
        rows,
        key=lambda row: (row['balanced_macro_f1'], row['raw_macro_f1'], -row['std_fold_balanced_macro_f1']),
        reverse=True,
    )


## Compare Two Preprocessing Designs Across Four Bands

This block runs every required frequency band under both preprocessing designs. The ranking uses balanced Macro-F1 first and raw Macro-F1 as a tie-breaker, because balanced assignment can saturate on such a small validation set.


In [ ]:
all_results = []
for design_name, design in PREPROCESSING_DESIGNS.items():
    print(f'Preprocessing design: {design_name}')
    for band_name, band in BANDS.items():
        rows = evaluate_design_band(design_name, design, band_name, band)
        best_row = rank_results(rows)[0]
        all_results.append(best_row)
        print(
            f"  {band_name:20s} C={best_row['c']:<5} "
            f"raw_f1={best_row['raw_macro_f1']:.4f} "
            f"balanced_f1={best_row['balanced_macro_f1']:.4f} "
            f"std={best_row['std_fold_balanced_macro_f1']:.4f}"
        )

results_ranked = rank_results(all_results)
print('\nRanked summary:')
for i, row in enumerate(results_ranked, start=1):
    print(
        f"{i}. {row['design']} | {row['band']} | C={row['c']} | "
        f"raw_f1={row['raw_macro_f1']:.4f} | balanced_f1={row['balanced_macro_f1']:.4f} | "
        f"balanced_acc={row['balanced_accuracy']:.4f}"
    )


## Final Model And Submission

The final model is selected from local validation only. It is then trained on all labeled training trials and used to predict the official test set. The output file is ignored by git and can be uploaded to Kaggle manually.


In [ ]:
def train_final_and_predict(selected: dict) -> tuple[list[tuple[int, int]], dict]:
    design = PREPROCESSING_DESIGNS[selected['design']]
    band = BANDS[selected['band']]

    x_fit, x_test = fit_preprocess_transform(x_train_raw, x_test_raw, sfreq_train, band, design)
    csp_filters = fit_csp_filters(x_fit, y_train)
    train_features = extract_features(x_fit, csp_filters)
    test_features = extract_features(x_test, csp_filters)

    classifier = build_classifier(float(selected['c']))
    classifier.fit(train_features, y_train)
    probabilities = classifier.predict_proba(test_features)
    pred = balanced_assignment(np.log(np.clip(probabilities, 1e-12, 1.0)))
    rows = [(int(sample_id), int(label)) for sample_id, label in zip(test_ids, pred)]
    metadata = {
        'selected_design': selected['design'],
        'selected_band': selected['band'],
        'selected_c': float(selected['c']),
        'test_label_counts': dict(zip(*np.unique(pred, return_counts=True))),
    }
    return rows, metadata


def write_submission(rows: list[tuple[int, int]], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'label'])
        writer.writerows(rows)


selected_result = results_ranked[0]
submission_rows, final_metadata = train_final_and_predict(selected_result)
output_path = Path('task1_submission.csv')
write_submission(submission_rows, output_path)

print('Selected final pipeline:')
for key, value in final_metadata.items():
    print(f'  {key}: {value}')
print(f'Wrote {len(submission_rows)} rows to {output_path}')


## Report Notes

Use these points in the report:

- We compare two preprocessing designs: full-trial global z-score vs. motor-window per-channel z-score.
- Every design is evaluated on all four required bands: 8-13, 13-30, 4-40, and 70-125 Hz.
- The validation protocol is 4-fold stratified validation on the released training set.
- The best result favors the motor-window per-channel design with the broad 4-40 Hz band. This suggests that, for within-subject classification, keeping a broad motor-related band can help because subject-specific channel structure is consistent between train and test.
- This contrasts with Task 2, where cross-subject generalization preferred narrower and more conservative bands.
